<a href="https://colab.research.google.com/github/paulatul9015/Word-Prediction/blob/main/LSTM_Word_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import pickle

import numpy as np
import pandas as pd
from nltk.tokenize import RegexpTokenizer

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Activation
from tensorflow.keras.optimizers import RMSprop

In [ ]:
text_df = pd.read_csv("converted.csv", on_bad_lines='warn')

/tmp/ipython-input-2197031682.py:1: ParserWarning: Skipping line 3: expected 33 fields, saw 37
Skipping line 4: expected 33 fields, saw 46
Skipping line 5: expected 33 fields, saw 42
Skipping line 6: expected 33 fields, saw 37
Skipping line 7: expected 33 fields, saw 44
Skipping line 9: expected 33 fields, saw 40
Skipping line 10: expected 33 fields, saw 39
Skipping line 11: expected 33 fields, saw 42
Skipping line 12: expected 33 fields, saw 34
Skipping line 14: expected 33 fields, saw 45
Skipping line 15: expected 33 fields, saw 42
Skipping line 17: expected 33 fields, saw 38
Skipping line 18: expected 33 fields, saw 36
Skipping line 19: expected 33 fields, saw 48
Skipping line 20: expected 33 fields, saw 46
Skipping line 21: expected 33 fields, saw 50
Skipping line 24: expected 33 fields, saw 44
Skipping line 25: expected 33 fields, saw 44
Skipping line 26: expected 33 fields, saw 37
Skipping line 27: expected 33 fields, saw 38
Skipping line 28: expected 33 fields, saw 43
Skipping l

In [ ]:
text = list(text_df.iloc[:, 0].astype(str).values)
text = " ".join(text)

In [ ]:
partial_text = text[:1000000]

In [ ]:
tokenizer = RegexpTokenizer(r"\w+")
tokens = tokenizer.tokenize(partial_text.lower())

In [ ]:
from enum import unique
unique_tokens = np.unique(tokens)
unique_token_index = {token: idx for idx, token in enumerate(unique_tokens)}

In [ ]:
n_words = 100
input_words = []
next_words = []

for i in range(len(tokens) - n_words):
    input_words.append(tokens[i:i + n_words])
    next_words.append(tokens[i + n_words])

In [ ]:
x = np.zeros((len(input_words), n_words, len(unique_tokens)), dtype=bool)
y = np.zeros((len(next_words), len(unique_tokens)), dtype=bool)

In [ ]:
for i , words in enumerate(input_words):
    for j, word in enumerate(words):
        x[i, j, unique_token_index[word]] = 1
    y[i, unique_token_index[next_words[i]]] = 1

In [ ]:
model = Sequential()
model.add(LSTM(512, input_shape=(n_words, len(unique_tokens)), return_sequences=True))
model.add(LSTM(512))
model.add(Dense(len(unique_tokens)))
model.add(Activation("softmax"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model.compile(
    optimizer=RMSprop(learning_rate=0.01),
    loss="categorical_crossentropy",
)

In [ ]:
model.save("mymodel.h5")

In [ ]:
model = load_model("mymodel.h5")

In [ ]:
def predict_next_word(input_text, n_best):
    input_text = input_text.lower()
    X = np.zeros((1, n_words, len(unique_tokens)))
    for i, word in enumerate(input_text.split()):
        X[0, i, unique_token_index[word]] = 1
    predictions = model.predict(X)[0]
    return np.argpartition(predictions, -n_best)[-n_best:]

In [ ]:
possible = predict_next_word("I will have ", 5)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 431ms/step


In [ ]:
print([unique_tokens[idx] for idx in possible])

[np.str_('yet'), np.str_('you'), np.str_('young'), np.str_('your'), np.str_('1')]


In [ ]:
def generate_text(input_text, text_length, creativity=3):
  word_sequence = input_text.split()
  current = 0
  for _ in range(text_length):
    # Corrected: ' '.join instead of join() and correct input to tokenizer.tokenize
    sub_sequence = " ".join(tokenizer.tokenize(" ".join(word_sequence[current:current+n_words]))[-n_words:])
    try:
      choice = unique_tokens[random.choice(predict_next_word(sub_sequence, creativity))]
    except:
      choice = random.choice(unique_tokens)
    word_sequence.append(choice)
    current += 1
  return "  ".join(word_sequence)

In [ ]:
generate_text("I will have ", 100,5)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 238ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 254ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 219ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 240ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 245ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 251ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 

'I  will  have  you  yet  young  1  young  young  1  yet  your  1  young  1  young  you  you  you  your  yet  you  you  your  young  your  you  your  you  you  you  young  you  1  young  your  young  young  you  your  you  your  young  young  you  young  you  young  your  yet  young  young  you  your  your  you  your  yet  yet  1  young  your  yet  young  1  yet  young  young  young  1  your  yet  your  young  you  1  you  you  yet  your  yet  you  yet  1  you  you  young  young  1  you  your  yet  yet  your  young  yet  young  yet  you  1  you  young  young'

In [ ]:
model = load_model("mymodel.h5")